In [ ]:
# Standard Data Science Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
import pyarrow
import datetime

# Tiingo API
import tiingo
from tiingo import TiingoClient

# Web Requests and Querying
import requests
import  json
import os
#import pandas_datareader as web
#from aquarel import load_theme

from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

True

In [ ]:
# TIINGO API Parameters

def make_client() -> TiingoClient:
    key = os.getenv("TIINGO_API_KEY")
    if not key:
        raise RuntimeError("TIINGO_API_KEY not set")
    return TiingoClient({"session": True, "api_key": key})

In [ ]:
# API Call Parameters

TICKER = "SPY"
LOOKBACK_DAYS = 92               # ~3 months of calendar days
INTRADAY_FREQ = "30min"          # 1min | 5min | 15min | 30min | 1hour
CHUNK_DAYS = 20                  # safe window per IEX request at 30min;
                                 # drop to ~5 if using 1min
REQUEST_SLEEP_S = 0.25
 
OUT_INTRADAY = "/data/spy_intraday.parquet"
OUT_DAILY    = "/data/spy_daily.parquet"
 

In [ ]:
# Helpers

def daterange_chunks(start: datetime, end: datetime, chunk_days: int) -> Iterator[tuple[datetime, datetime]]:
    cur = start
    while cur <= end:
        nxt = min(cur + timedelta(days=chunk_days - 1), end)
        yield cur, nxt
        cur = nxt + timedelta(days=1)
 
 
def fetch_intraday(
    client: TiingoClient,
    ticker: str,
    start: datetime,
    end: datetime,
    freq: str,
) -> pd.DataFrame:
    """Pull IEX intraday bars in chunks and concatenate."""
    frames: list[pd.DataFrame] = []
    for c_start, c_end in daterange_chunks(start, end, CHUNK_DAYS):
        rows = client.get_ticker_price(
            ticker,
            startDate=c_start.strftime("%Y-%m-%d"),
            endDate=c_end.strftime("%Y-%m-%d"),
            frequency=freq,
            fmt="json",
        )
        if not rows:
            continue
        frames.append(pd.DataFrame(rows))
        time.sleep(REQUEST_SLEEP_S)
 
    if not frames:
        raise RuntimeError(f"No intraday data returned for {ticker}")
 
    out = pd.concat(frames, ignore_index=True)
    out["date"] = pd.to_datetime(out["date"], utc=True)
    out = (
        out.drop_duplicates(subset="date")
           .sort_values("date")
           .reset_index(drop=True)
    )
    out["date_et"] = out["date"].dt.tz_convert("America/New_York")
    out["session_date"] = out["date_et"].dt.date
    return out
 
 
def fetch_daily(
    client: TiingoClient,
    ticker: str,
    start: datetime,
    end: datetime,
) -> pd.DataFrame:
    """
    Pull EOD daily bars via get_dataframe. This hits the standard endpoint
    (not IEX), so `volume` is consolidated tape and adjusted columns are
    available.
    """
    df = client.get_dataframe(
        ticker,
        startDate=start.strftime("%Y-%m-%d"),
        endDate=end.strftime("%Y-%m-%d"),
        frequency="daily",
    )
    # get_dataframe returns a DatetimeIndex named 'date' with columns:
    # open, high, low, close, volume, adjOpen, adjHigh, adjLow, adjClose,
    # adjVolume, divCash, splitFactor
    df = df.reset_index().rename(columns={"index": "date"})
    # Normalize to ET session date for easy joining against intraday.
    df["date"] = pd.to_datetime(df["date"], utc=True).dt.tz_convert("America/New_York")
    df["session_date"] = df["date"].dt.date
    return df.sort_values("date").reset_index(drop=True)